# Part 1

In [ ]:
# ============================================
# INSTALL REQUIRED PACKAGES
# ============================================

!pip install -q transformers accelerate bitsandbytes
!pip install -q langdetect tqdm pandas numpy requests

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import os
import re
import json
import time
import torch
import pandas as pd
import numpy as np

from tqdm import tqdm
from langdetect import detect, LangDetectException

import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded")

In [ ]:
# ============================================
# GOOGLE DRIVE
# ============================================

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ============================================
# PATH CONFIGURATION
# ============================================

BASE_PATH = "/content/drive/MyDrive/mbti-tune"

CLEANED_PATH = f"{BASE_PATH}/data/cleaned"

THEME_PATH = f"{BASE_PATH}/data/themes"

os.makedirs(THEME_PATH, exist_ok=True)


# Previous notebook output
PLAYLIST_FILE = (
    f"{CLEANED_PATH}/classifier/"
    "balanced_10song_playlists.csv"
)


# New files
UNIQUE_SONG_FILE = (
    f"{THEME_PATH}/unique_songs.csv"
)

LYRICS_FILE = (
    f"{THEME_PATH}/song_lyrics.csv"
)

SONG_THEME_FILE = (
    f"{THEME_PATH}/song_theme_features.csv"
)

PLAYLIST_THEME_FILE = (
    f"{THEME_PATH}/playlist_theme_features.csv"
)


print("Paths configured")
print(THEME_PATH)

In [ ]:
# ============================================
# THEME DEFINITIONS
# ============================================

THEMES = [
    "romance",
    "heartbreak",
    "loneliness",
    "hope",
    "confidence",
    "sadness",
    "happiness",
    "celebration",
    "adventure",
    "freedom",
    "self_discovery",
    "nostalgia",
    "friendship",
    "ambition",
    "rebellion"
]


MOODS = [
    "happy",
    "sad",
    "energetic",
    "calm",
    "dark"
]


print("Themes:")
for t in THEMES:
    print("-", t)

print("\nMoods:")
for m in MOODS:
    print("-", m)

In [ ]:
# ============================================
# LOAD AUGMENTED DATASET
# ============================================


if not os.path.exists(PLAYLIST_FILE):
    raise FileNotFoundError(
        "balanced_10song_playlists.csv not found"
    )


df_playlists = pd.read_csv(
    PLAYLIST_FILE
)


print("="*60)
print("AUGMENTED DATASET")
print("="*60)

print("Shape:")
print(df_playlists.shape)

print()

print("Columns:")
print(df_playlists.columns.tolist()[:20])

print()

print(
    "Playlists:",
    df_playlists["sub_playlist_id"].nunique()
)

print(
    "MBTI types:",
    df_playlists["mbti"].nunique()
)


df_playlists.head()

In [ ]:
# ============================================
# EXTRACT SONG RECORDS
# ============================================


song_records = []


for _, row in df_playlists.iterrows():

    playlist_id = row["sub_playlist_id"]
    mbti = row["mbti"]


    for i in range(1,11):

        song_col = f"song_{i}_name"
        artist_col = f"song_{i}_artist"
        track_col = f"song_{i}_track_id"


        if song_col in row:

            song = row[song_col]
            artist = row[artist_col]
            track_id = row.get(track_col,"")


            if (
                pd.notna(song)
                and pd.notna(artist)
                and song != ""
                and artist != ""
            ):

                song_records.append({

                    "playlist_id":playlist_id,
                    "mbti":mbti,
                    "song_index":i,
                    "song":song,
                    "artist":artist,
                    "track_id":track_id

                })


df_song_records = pd.DataFrame(song_records)


print(
    "Playlist-song records:",
    len(df_song_records)
)


print(
    "Unique songs:",
    df_song_records[["song","artist"]]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
# ============================================
# UNIQUE SONGS
# ============================================


df_unique_songs = (
    df_song_records[
        [
            "song",
            "artist",
            "track_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


df_unique_songs["song_id"] = (
    "song_"
    +
    df_unique_songs.index.astype(str)
)



print(
    "Unique songs:",
    len(df_unique_songs)
)


df_unique_songs.to_csv(
    UNIQUE_SONG_FILE,
    index=False
)


print(
    "Saved:",
    UNIQUE_SONG_FILE
)


df_unique_songs.head()

In [ ]:
# ============================================
# CHECK GPU
# ============================================

print(torch.cuda.get_device_name(0))

In [ ]:
# ============================================
# LOAD QWEN MODEL
# ============================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"



quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)


model.eval()


print("Qwen loaded successfully")

In [ ]:
# ============================================
# TEST MODEL
# ============================================


messages = [
    {
        "role":"user",
        "content":
        "Explain what a romantic song is in one sentence."
    }
]


text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)


inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)



with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2
    )



response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)


print(response)

## Part 2

In [ ]:
# ============================================
# IMPORT LYRICS DEPENDENCIES
# ============================================

import requests
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed


print("Lyrics libraries loaded")

In [ ]:
# ============================================
# LOAD UNIQUE SONG DATA
# ============================================


df_unique_songs = pd.read_csv(
    UNIQUE_SONG_FILE
)


print(
    "Unique songs:",
    len(df_unique_songs)
)


df_unique_songs.head()

In [ ]:
# ============================================
# CLEANING FUNCTIONS
# ============================================


def clean_song_name(song):

    if pd.isna(song):
        return ""


    song = str(song)


    # Remove brackets
    song = re.sub(
        r"\([^)]*\)",
        "",
        song
    )


    song = re.sub(
        r"\[[^\]]*\]",
        "",
        song
    )


    # Remove feat information
    song = re.sub(
        r"(feat\.|ft\.|featuring).*",
        "",
        song,
        flags=re.IGNORECASE
    )


    song = " ".join(song.split())


    return song.strip()



def clean_artist_name(artist):

    if pd.isna(artist):
        return ""


    artist = str(artist)


    artist = re.sub(
        r"(feat\.|ft\.|featuring).*",
        "",
        artist,
        flags=re.IGNORECASE
    )


    artist = " ".join(
        artist.split()
    )


    return artist.strip()

In [ ]:
# ============================================
# LRCLIB API
# ============================================


def get_lrclib_lyrics(song, artist):


    url = "https://lrclib.net/api/get"


    params = {

        "track_name":
            clean_song_name(song),

        "artist_name":
            clean_artist_name(artist)

    }


    try:

        response = requests.get(
            url,
            params=params,
            timeout=15
        )


        if response.status_code == 200:

            data = response.json()


            lyrics = data.get(
                "plainLyrics"
            )


            if lyrics:

                return {

                    "lyrics":lyrics,

                    "source":"lrclib"

                }


    except Exception:

        pass


    return None

In [ ]:
# ============================================
# LYRICS.OVH BACKUP
# ============================================


def get_ovh_lyrics(song, artist):


    artist = clean_artist_name(artist)
    song = clean_song_name(song)


    url = (
        "https://api.lyrics.ovh/v1/"
        f"{artist}/{song}"
    )


    try:

        response = requests.get(
            url,
            timeout=15
        )


        if response.status_code == 200:

            data=response.json()


            lyrics=data.get(
                "lyrics"
            )


            if lyrics:

                return {

                    "lyrics":lyrics,

                    "source":"lyrics.ovh"

                }


    except Exception:

        pass


    return None

In [ ]:
# ============================================
# COMBINED FETCH
# ============================================


def fetch_song_lyrics(song, artist):


    result = get_lrclib_lyrics(
        song,
        artist
    )


    if result:

        return result



    result = get_ovh_lyrics(
        song,
        artist
    )


    if result:

        return result



    return {

        "lyrics":"",
        "source":"none"

    }

In [ ]:
# ============================================
# TEST
# ============================================


test = fetch_song_lyrics(
    "Bohemian Rhapsody",
    "Queen"
)


print(test["source"])

print(
    test["lyrics"][:300]
)

In [ ]:
# ============================================
# PROGRESS SETTINGS
# ============================================


if os.path.exists(LYRICS_FILE):

    df_existing = pd.read_csv(
        LYRICS_FILE
    )


    completed = set(
        zip(
            df_existing.song,
            df_existing.artist
        )
    )


    print(
        "Existing:",
        len(completed)
    )


else:

    df_existing = pd.DataFrame()

    completed=set()


print(
    "Remaining:",
    len(df_unique_songs)-len(completed)
)

In [ ]:
# ============================================
# PARALLEL FETCHING
# ============================================


results=[]


lock=threading.Lock()



def process_song(row):


    song=row["song"]
    artist=row["artist"]


    result = fetch_song_lyrics(
        song,
        artist
    )


    return {

        "song":song,

        "artist":artist,

        "track_id":row["track_id"],

        "lyrics":
            result["lyrics"],

        "source":
            result["source"],

        "has_lyrics":
            len(result["lyrics"]) > 50

    }

In [ ]:
# ============================================
# RUN FETCH
# ============================================


songs_to_process=[]


for _,row in df_unique_songs.iterrows():

    key=(
        row["song"],
        row["artist"]
    )


    if key not in completed:

        songs_to_process.append(row)



print(
    "Songs to fetch:",
    len(songs_to_process)
)

In [ ]:
# ============================================
# FETCH WITH THREADS
# ============================================


new_results=[]


with ThreadPoolExecutor(
    max_workers=8
) as executor:


    futures=[

        executor.submit(
            process_song,
            row
        )

        for _,row
        in pd.DataFrame(songs_to_process).iterrows()

    ]


    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        new_results.append(
            future.result()
        )


        # save every 100 songs

        if len(new_results)%100==0:


            temp=pd.DataFrame(
                new_results
            )


            if len(df_existing)>0:

                temp=pd.concat(
                    [
                        df_existing,
                        temp
                    ],
                    ignore_index=True
                )


            temp.to_csv(
                LYRICS_FILE,
                index=False
            )


            print(
                "Saved:",
                len(temp)
            )

In [ ]:
# ============================================
# FINAL SAVE
# ============================================


df_lyrics = pd.DataFrame(
    new_results
)


if len(df_existing)>0:

    df_lyrics=pd.concat(
        [
            df_existing,
            df_lyrics
        ],
        ignore_index=True
    )


df_lyrics.to_csv(
    LYRICS_FILE,
    index=False
)



print("="*60)

print(
    "Lyrics dataset created"
)


print(
    "Total songs:",
    len(df_lyrics)
)


print(
    "With lyrics:",
    df_lyrics.has_lyrics.sum()
)


print(
    "Success rate:",
    round(
        df_lyrics.has_lyrics.mean()*100,
        2
    ),
    "%"
)

print("="*60)